# SkinMamba — end-to-end skin lesion segmentation on ISIC 2018 (Task 1) · Google Colab

**Pipeline** (mirrors the DuAT reference notebook, re-targeted to Colab + SkinMamba)

| # | Stage | What happens |
|---|-------|--------------|
| 1 | Data | `!wget` straight from the ISIC challenge page → unzip → pair images/masks (2594 train + 100 val + 1000 official test) |
| 2 | **Medical image processing** (offline, cached on Drive) | dermoscope FOV mask · DullRazor-style hair removal with border-safe flank test · Shades-of-Gray colour constancy · conditional denoising · CLAHE on LAB-L |
| 3 | Augmentation (on the fly) | flips / rot90 / affine / elastic · LAB colour transfer between lesions (skin-tone robustness) · HSV + brightness · median/motion blur · coarse dropout |
| 4 | Model | **SkinMamba** (SRSSB = VSSB/SS2D + Scale-Mixed FFN, FBGM frequency boundary guidance) written from the paper on top of the `mamba-ssm` selective-scan kernel — *not* cloned from the authors' repo |
| 5 | Loss | Focal-Tversky + edge-weighted Structure loss |
| 6 | Training | AdamW, warm-up + cosine, mixed precision, early stopping, checkpoints on Google Drive, **auto-resume after Colab disconnects** |
| 7 | Inference | flip-TTA → FOV gating, morphology, hole filling, component filtering, optional DenseCRF |
| 8 | Evaluation / export | paper metrics (mIoU, DSC, Acc, Spe, Sen) + official ISIC *thresholded Jaccard* at original resolution; PNG masks zipped to Drive |

**How to run:** `Runtime ▸ Change runtime type ▸ GPU` (T4 works, L4/A100 are faster) → run all cells top to bottom.
First run downloads ~13 GB and builds a preprocessed cache on Drive; later sessions skip that and take seconds.

Paper: *SkinMamba: A Precision Skin Lesion Segmentation Architecture with Cross-Scale Global State Modeling and Frequency Boundary Guidance* (Zou et al., 2024, arXiv:2409.10890).

In [ ]:
# ============================ 1. ENVIRONMENT & CONFIG ============================
import os, sys, math, json, time, random, shutil, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)

try:
    import google.colab  # noqa: F401
except ImportError:
    raise RuntimeError("This notebook targets Google Colab only (Drive mount + Colab GPU runtime).")

import torch
assert torch.cuda.is_available(), "No GPU found: Runtime > Change runtime type > pick a GPU (T4 / L4 / A100)."
_prop = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_GB, GPU_CC = _prop.name, _prop.total_memory / 2**30, torch.cuda.get_device_capability(0)
print(f"GPU {GPU_NAME} | {GPU_GB:.1f} GB | sm_{GPU_CC[0]}{GPU_CC[1]} | torch {torch.__version__} | CUDA {torch.version.cuda} | Python {sys.version.split()[0]}")

from google.colab import drive
drive.mount("/content/drive")

# ----------------------------------- paths -----------------------------------
DRIVE_DIR = Path("/content/drive/MyDrive/SkinMamba_ISIC2018"); DRIVE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR  = DRIVE_DIR / "checkpoints";                       CKPT_DIR.mkdir(exist_ok=True)
DATA_DIR  = Path("/content/data");                           DATA_DIR.mkdir(exist_ok=True)
PRED_DIR  = Path("/content/predictions")

# ----------------------------- experiment settings ----------------------------
SEED = 42
IMG_SIZE = 224                  # paper: 224x224 (must be divisible by 32)
VAL_FRACTION = 0.10             # held-out part of the train pool used for model selection
USE_OFFICIAL_VAL_IN_TRAIN = True  # train(2594)+official val(100) = 2694 images, as in the paper; test = official 1000

# model (paper defaults)
BASE_CHANNELS, DEPTHS, D_STATE, EXPAND = 16, (1, 1, 1, 1, 1), 16, 2
SKIP_MODE = "add"               # Sec. 3.1 of the paper: skip connections are plain additions ("concat" also supported)
FBGM_DEPTH = 1

# optimisation
MAX_EPOCHS, WARMUP_EPOCHS = 100, 3   # paper: 300 epochs (too long for one Colab session; training auto-resumes)
LR, WEIGHT_DECAY = 1e-3, 1e-4        # paper: AdamW, lr 1e-3, wd 1e-4, cosine annealing
EARLY_STOP_PATIENCE = 25
FT_WEIGHT = 0.6                      # loss = 0.6 * FocalTversky + 0.4 * StructureLoss

# hardware-dependent defaults (override manually if you hit OOM)
AMP_DTYPE = torch.bfloat16 if GPU_CC[0] >= 8 else torch.float16
PRECISION = "bf16-mixed" if GPU_CC[0] >= 8 else "16-mixed"      # set "32-true" if you ever see NaN losses
BATCH_SIZE = 8 if GPU_GB < 20 else (16 if GPU_GB < 30 else 32)
ACCUM = max(1, 16 // BATCH_SIZE)                                 # keeps the effective batch >= 16
NUM_WORKERS = min(4, os.cpu_count() or 2)

# misc switches
USE_WANDB = False                    # optional; the key is read from Colab *Secrets* (never paste keys into a notebook)
BUILD_MAMBA_FROM_SOURCE = True       # only used if no prebuilt mamba-ssm wheel matches this runtime (10-25 min)
ALLOW_SLOW_FALLBACK = False          # allow the (very slow, memory hungry) pure-PyTorch scan if mamba-ssm is unavailable
CLEAN_RAW_AFTER_CACHE = True         # delete the ~13 GB of raw zips/images once the preprocessed cache is on Drive

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
print(f"batch {BATCH_SIZE} x accum {ACCUM} | precision {PRECISION} | workers {NUM_WORKERS} | Drive dir: {DRIVE_DIR}")

In [ ]:
# ============================ 2. DEPENDENCIES ============================
# Colab already ships torch, numpy, scipy, opencv, scikit-image, scikit-learn, pandas, matplotlib, tqdm.
!pip install -q pytorch-lightning "albumentations==2.0.8" einops packaging ninja

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from PIL import Image
from tqdm.auto import tqdm
from scipy.ndimage import binary_fill_holes
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger

cv2.setNumThreads(0)                                   # avoid CPU oversubscription with DataLoader / process pools
pl.seed_everything(SEED, workers=True)
print("albumentations", A.__version__, "| lightning", pl.__version__, "| opencv", cv2.__version__)

In [ ]:
# ============================ 3. MAMBA SELECTIVE-SCAN BACKEND ============================
# SkinMamba's SS2D needs Mamba's selective-scan CUDA kernel (library `mamba-ssm`). Prebuilt wheels exist only for
# specific torch / CUDA / Python combinations, so this cell (1) tries to import it, (2) looks up a matching wheel on the
# official GitHub releases, (3) otherwise compiles from source, (4) otherwise falls back to a pure-PyTorch scan.
import importlib, urllib.request

def selective_scan_chunked(u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False,
                           return_last_state=False, chunk=16):
    """Pure-PyTorch selective scan, same call signature as mamba_ssm (z / return_last_state unsupported).
    Exact chunked 'segsum' formulation (as in Mamba-2's SSD) - verified against the naive recurrence (max error 1e-13).
    Memory hungry and slow: only meant for debugging with tiny images.
    u, delta: (b,d,l)   A: (d,n)   B, C: (b,g,n,l)   D: (d,)"""
    assert z is None and not return_last_state
    b, d, l = u.shape; n = A.shape[1]; g = B.shape[1]; dg = d // g
    out_dtype = u.dtype
    u, delta, A, B, C = u.float(), delta.float(), A.float(), B.float(), C.float()
    if delta_bias is not None:
        delta = delta + delta_bias.float()[None, :, None]
    if delta_softplus:
        delta = F.softplus(delta)
    pad = (-l) % chunk
    if pad:                                   # delta = 0 -> identity step, u = 0 -> no input: state is unchanged
        u, delta = F.pad(u, (0, pad)), F.pad(delta, (0, pad))
        B, C = F.pad(B, (0, pad)), F.pad(C, (0, pad))
    L = l + pad
    u_g, dt_g, A_g = u.reshape(b, g, dg, L), delta.reshape(b, g, dg, L), A.reshape(g, dg, n)
    tri = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=u.device))[None, None, None, :, :, None]
    h = u.new_zeros(b, g, dg, n); ys = []
    for c in range(L // chunk):
        sl = slice(c * chunk, (c + 1) * chunk)
        dt, uu, Bc, Cc = dt_g[..., sl], u_g[..., sl], B[..., sl], C[..., sl]
        cum = (dt.unsqueeze(-1) * A_g[None, :, :, None, :]).cumsum(dim=3)               # (b,g,dg,T,n) log-decay
        dBu = (dt * uu).unsqueeze(-1) * Bc.permute(0, 1, 3, 2).unsqueeze(2)             # (b,g,dg,T,n)
        seg = (cum.unsqueeze(4) - cum.unsqueeze(3)).masked_fill(~tri, float("-inf"))    # [t,s] = cum_t - cum_s
        h_all = torch.einsum("bgdtsn,bgdsn->bgdtn", seg.exp(), dBu) + h.unsqueeze(3) * cum.exp()
        ys.append(torch.einsum("bgdtn,bgnt->bgdt", h_all, Cc))
        h = h_all[:, :, :, -1]
    y = torch.cat(ys, dim=-1).reshape(b, d, L)[:, :, :l]
    if D is not None:
        y = y + D.float()[None, :, None] * u[:, :, :l]
    return y.to(out_dtype)

def _load_cuda_scan():
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    return selective_scan_fn

def _gh_json(url):
    req = urllib.request.Request(url, headers={"Accept": "application/vnd.github+json", "User-Agent": "colab-skinmamba"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.load(r)

def find_mamba_wheel():
    tv = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    py = f"cp{sys.version_info.major}{sys.version_info.minor}"
    abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
    want = f"+cu{torch.version.cuda.split('.')[0]}torch{tv}cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
    print("looking for a prebuilt wheel ending in:", want)
    for rel in _gh_json("https://api.github.com/repos/state-spaces/mamba/releases?per_page=8"):
        for page in (1, 2, 3):
            assets = _gh_json(f"https://api.github.com/repos/state-spaces/mamba/releases/{rel['id']}/assets?per_page=100&page={page}")
            if not assets:
                break
            for a in assets:
                if a["name"].startswith("mamba_ssm-") and a["name"].endswith(want):
                    return a["browser_download_url"]
    return None

def setup_mamba():
    try:
        return _load_cuda_scan(), "mamba_ssm CUDA kernel"
    except Exception as e:
        print("mamba_ssm not importable yet:", repr(e)[:160])
    try:
        url = find_mamba_wheel()
    except Exception as e:
        print("wheel lookup failed:", repr(e)[:160]); url = None
    if url:
        print("installing prebuilt wheel:", url)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", url], check=False)
    elif BUILD_MAMBA_FROM_SOURCE:
        print("no prebuilt wheel for this runtime -> compiling from source (10-25 min, one-off per session) ...")
        os.environ.setdefault("MAX_JOBS", "2")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wheel", "setuptools"], check=False)
        subprocess.run([sys.executable, "-m", "pip", "install", "mamba-ssm", "--no-build-isolation"], check=False)
    importlib.invalidate_caches()
    try:
        return _load_cuda_scan(), "mamba_ssm CUDA kernel"
    except Exception as e:
        print("!! mamba_ssm is still unavailable:", repr(e)[:200])
        return None, "pure-PyTorch fallback (very slow)"

selective_scan_fn, MAMBA_BACKEND = setup_mamba()
if selective_scan_fn is None:
    selective_scan_fn = selective_scan_chunked
print("SS2D backend ->", MAMBA_BACKEND)

In [ ]:
# ============================ 4. DATASET (wget from the ISIC challenge page) ============================
# https://challenge.isic-archive.com/data/#2018  ->  Task 1 (lesion boundary segmentation)
ISIC = "https://isic-archive.s3.amazonaws.com/challenges/2018"
CACHE_FILE = DRIVE_DIR / f"isic2018_task1_cache_{IMG_SIZE}.npz"
NEED_RAW = not CACHE_FILE.exists()
print("preprocessed cache on Drive:", CACHE_FILE.exists(),
      "-> skipping the big downloads" if not NEED_RAW else "-> downloading and preprocessing (first run only)")

# small files are always fetched: the test ground truth is needed for full-resolution evaluation
!wget -c -q --show-progress "{ISIC}/ISIC2018_Task1_Test_GroundTruth.zip" -P {DATA_DIR}
!unzip -q -n {DATA_DIR}/ISIC2018_Task1_Test_GroundTruth.zip -d {DATA_DIR}

if NEED_RAW:
    !wget -c -q --show-progress "{ISIC}/ISIC2018_Task1_Training_GroundTruth.zip" -P {DATA_DIR}
    !wget -c -q --show-progress "{ISIC}/ISIC2018_Task1_Validation_GroundTruth.zip" -P {DATA_DIR}
    !wget -c -q --show-progress "{ISIC}/ISIC2018_Task1-2_Validation_Input.zip" -P {DATA_DIR}
    !wget -c -q --show-progress "{ISIC}/ISIC2018_Task1-2_Test_Input.zip" -P {DATA_DIR}                  # 2.2 GB
    !wget -c -q --show-progress "{ISIC}/ISIC2018_Task1-2_Training_Input.zip" -P {DATA_DIR}              # 10.4 GB
    for z in ["ISIC2018_Task1_Training_GroundTruth", "ISIC2018_Task1_Validation_GroundTruth",
              "ISIC2018_Task1-2_Validation_Input", "ISIC2018_Task1-2_Test_Input", "ISIC2018_Task1-2_Training_Input"]:
        !unzip -q -n {DATA_DIR}/{z}.zip -d {DATA_DIR}

print(sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
# ============================ 5. PAIR IMAGES AND MASKS ============================
TEST_GT_DIR = DATA_DIR / "ISIC2018_Task1_Test_GroundTruth"

def pairs_from(img_dir, gt_dir):
    """[(image_path, mask_path)] matched by ISIC id (ISIC_0000000.jpg <-> ISIC_0000000_segmentation.png)."""
    gts = {p.name[:-len("_segmentation.png")]: p for p in Path(gt_dir).rglob("ISIC_*_segmentation.png")}
    return [(p, gts[p.stem]) for p in sorted(Path(img_dir).rglob("ISIC_*.jpg")) if p.stem in gts]

if NEED_RAW:
    train_pairs = pairs_from(DATA_DIR / "ISIC2018_Task1-2_Training_Input",   DATA_DIR / "ISIC2018_Task1_Training_GroundTruth")
    val_pairs   = pairs_from(DATA_DIR / "ISIC2018_Task1-2_Validation_Input", DATA_DIR / "ISIC2018_Task1_Validation_GroundTruth")
    test_pairs  = pairs_from(DATA_DIR / "ISIC2018_Task1-2_Test_Input",       TEST_GT_DIR)
    print(f"train {len(train_pairs)} | val {len(val_pairs)} | test {len(test_pairs)}   (expected 2594 | 100 | 1000)")
    assert (len(train_pairs), len(val_pairs), len(test_pairs)) == (2594, 100, 1000), "unexpected file counts - check the unzip step"

In [ ]:
# ============================ 6. MEDICAL IMAGE PROCESSING (dermoscopy) ============================
# ---------------------------------------------------------------------------
# Medical (dermoscopy) image processing — applied ONCE offline, then cached.
#   1. Field-of-view (FOV) mask      -> ignores the black dermoscope vignette
#   2. DullRazor-style hair removal  -> multi-orientation black-hat + elongation filter + inpainting
#   3. Shades-of-Gray colour constancy (Minkowski p=6, FOV only) -> removes illuminant colour casts
#   4. Conditional denoising         -> 3x3 median blur only when Immerkaer noise estimate is high
#   5. CLAHE on the LAB L-channel    -> local contrast of lesion borders without colour shifts
# ---------------------------------------------------------------------------
from scipy.ndimage import binary_fill_holes

def odd(x):
    x = int(round(x)); return x if x % 2 == 1 else x + 1

def compute_fov_mask(img, thr=12):
    """Binary mask (uint8, 1 = visible skin) of the circular/rounded dermoscope field of view."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    m = (gray > thr).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n <= 1:
        return np.ones_like(m)
    m = (lab == 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))).astype(np.uint8)
    m = binary_fill_holes(m).astype(np.uint8)       # dark lesions inside the FOV must stay inside
    return m if m.mean() > 0.20 else np.ones_like(m)  # safety: no plausible vignette -> full frame

def _line_kernel(length, angle_deg):
    k = np.zeros((length, length), np.uint8); c = length // 2
    dx, dy = np.cos(np.deg2rad(angle_deg)), np.sin(np.deg2rad(angle_deg))
    cv2.line(k, (int(round(c - dx * c)), int(round(c - dy * c))),
                (int(round(c + dx * c)), int(round(c + dy * c))), 1, 1)
    return k

def remove_hair(img, fov=None, thr=10, strong=22, min_frac=0.0005, max_frac=0.15, min_elong=5.0, ref_size=224):
    """DullRazor-style hair removal. Returns (clean_img, hair_mask, was_modified).
    Candidates come from a multi-orientation black-hat (ignoring the dermoscope vignette); then:
      * elongation index diag^2/area (>= 5): thin & long vs. blobs such as globules or bubbles (~2-2.5)
      * local flank test (per 16x16 tile): a hair has similar bright skin on BOTH sides, a lesion border has
        dark lesion on one side and skin on the other. Inpainting a border would paint skin over the lesion
        edge (bad for IoU), so tiles whose surroundings are bimodal are left untouched.
      * strength test: tiles must contain strong black-hat responses (median >= `strong`); the weak, thin
        ramps that curved lesion borders produce (~10-18) are never inpainted."""
    s = max(img.shape[:2]) / ref_size
    ksz = max(7, odd(11 * s))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    H, W = gray.shape
    bh = np.zeros_like(gray)
    for ang in (0, 30, 60, 90, 120, 150):
        bh = np.maximum(bh, cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, _line_kernel(ksz, ang)))
    cand = (bh > thr).astype(np.uint8)
    if fov is not None:                                    # ignore the black vignette and its rim
        cand &= cv2.erode(fov, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksz, ksz)))
    n, lab, st, _ = cv2.connectedComponentsWithStats(cand, connectivity=8)
    if n <= 1:
        return img, np.zeros_like(cand), False
    w, h, area = st[:, 2].astype(np.float32), st[:, 3].astype(np.float32), st[:, 4].astype(np.float32)
    elong = (w ** 2 + h ** 2) / np.maximum(area, 1)
    keep = (area >= 12 * s * s) & (elong >= min_elong) & (np.hypot(w, h) >= 1.2 * ksz)
    keep[0] = False
    geo = keep[lab].astype(np.uint8)                       # geometric hair candidates
    # ---- local flank test --------------------------------------------------------------------
    rad = max(3, int(round(4 * s))); T = max(12, int(round(16 * s)))
    k_in = np.ones((3, 3), np.uint8)
    k_out = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * rad + 1, 2 * rad + 1))
    mask = np.zeros_like(geo)
    for ty in range(0, H, T):
        for tx in range(0, W, T):
            core = geo[ty:ty + T, tx:tx + T]
            if core.sum() < 6:
                continue
            y0, x0, y1, x1 = max(0, ty - rad - 1), max(0, tx - rad - 1), min(H, ty + T + rad + 1), min(W, tx + T + rad + 1)
            win_geo = geo[y0:y1, x0:x1]
            here = np.zeros_like(win_geo); here[ty - y0:ty - y0 + core.shape[0], tx - x0:tx - x0 + core.shape[1]] = core
            ring = (cv2.dilate(here, k_out) > 0) & (cv2.dilate(win_geo, k_in) == 0)   # exclude every candidate
            g = gray[y0:y1, x0:x1]; rv = g[ring]
            if rv.size < 20:
                continue
            contrast = float(np.median(rv)) - float(np.median(g[here > 0]))
            spread = float(np.percentile(rv, 90) - np.percentile(rv, 10))
            strength = float(np.median(bh[y0:y1, x0:x1][here > 0]))
            if contrast >= 15 and spread <= 0.6 * contrast + 8 and strength >= strong:
                mask[ty:ty + T, tx:tx + T] = core
    frac = float(mask.mean())
    if frac < min_frac or frac > max_frac:           # no hair, or suspiciously much -> leave untouched
        return img, mask, False
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8))
    return cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA), mask, True

def shades_of_gray(img, fov=None, p=6):
    """Colour constancy (Finlayson & Trezzi). Illuminant = Minkowski-p mean over the FOV pixels."""
    x = img.astype(np.float32)
    sel = x[fov > 0] if fov is not None else x.reshape(-1, 3)
    if sel.shape[0] < 100:
        return img
    illum = np.power(np.mean(np.power(sel, p), axis=0), 1.0 / p)
    illum = illum / (np.linalg.norm(illum) + 1e-6)
    gain = np.clip(1.0 / (illum * np.sqrt(3.0) + 1e-6), 0.6, 1.6)
    return np.clip(x * gain, 0, 255).astype(np.uint8)

def estimate_noise_sigma(gray):
    """Immerkaer's fast noise standard-deviation estimate (gray levels)."""
    H, W = gray.shape
    M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    return float(np.sum(np.abs(cv2.filter2D(gray.astype(np.float32), -1, M))) * np.sqrt(0.5 * np.pi) / (6.0 * (W - 2) * (H - 2)))

def apply_clahe_lab(img, clip=2.0, tiles=8):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tiles, tiles)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def preprocess_dermoscopy(img, noise_thr=4.0):
    """img: HxWx3 uint8 RGB (already resized). Returns (processed_img, fov_mask, info_dict)."""
    fov = compute_fov_mask(img)
    out, hair_mask, had_hair = remove_hair(img, fov)
    out = shades_of_gray(out, fov)
    sigma = estimate_noise_sigma(cv2.cvtColor(out, cv2.COLOR_RGB2GRAY))
    denoised = sigma > noise_thr
    if denoised:
        out = cv2.medianBlur(out, 3)
    out = apply_clahe_lab(out)
    out = out * fov[..., None]                       # keep the vignette black
    return out, fov, dict(had_hair=had_hair, hair_frac=float(hair_mask.mean()), noise=sigma, denoised=denoised)

In [ ]:
# ============================ 7. BUILD (or LOAD) THE PREPROCESSED CACHE ============================
# Heavy, deterministic medical preprocessing is done ONCE here and stored on Drive; training then only needs
# cheap on-the-fly augmentation. Masks are resized with area interpolation + re-binarised.
S = IMG_SIZE

def load_resized_rgb(path, size):
    """Fast JPEG decode: PIL's draft mode decodes at reduced scale (DCT scaling) - big speed-up for 6000 px images."""
    im = Image.open(path); w0, h0 = im.size
    im.draft("RGB", (size * 2, size * 2))
    arr = cv2.resize(np.asarray(im.convert("RGB")), (size, size), interpolation=cv2.INTER_AREA)
    return arr, (h0, w0)

def _process_one(pair):
    ip, mp = pair
    raw, hw = load_resized_rgb(ip, S)
    m = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
    m = (cv2.resize(m, (S, S), interpolation=cv2.INTER_AREA) > 127).astype(np.uint8)
    out, fov, info = preprocess_dermoscopy(raw)
    return out, m, fov, hw, info["had_hair"], raw

def build_split(pairs, name):
    n = len(pairs)
    imgs = np.zeros((n, S, S, 3), np.uint8); masks = np.zeros((n, S, S), np.uint8); fovs = np.zeros((n, S, S), np.uint8)
    sizes = np.zeros((n, 2), np.int32); hair = np.zeros(n, bool); demo = []
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2, mp_context=mp.get_context("fork")) as ex:
        for i, (o, m, f, hw, hh, raw) in enumerate(tqdm(ex.map(_process_one, pairs, chunksize=8), total=n, desc=name)):
            imgs[i], masks[i], fovs[i], sizes[i], hair[i] = o, m, f, hw, hh
            if hh and len(demo) < 8:
                demo.append(raw)
    ids = np.array([p[0].stem for p in pairs])
    return dict(imgs=imgs, masks=masks, fovs=fovs, sizes=sizes, hair=hair, ids=ids,
                demo=np.array(demo, np.uint8).reshape(-1, S, S, 3))

if NEED_RAW:
    t0 = time.time()
    tv = build_split(train_pairs + (val_pairs if USE_OFFICIAL_VAL_IN_TRAIN else []), "train+val")
    te = build_split(test_pairs, "test")
    tmp = Path("/content/_cache_tmp.npz")
    np.savez(tmp, demo_raw=tv["demo"], **{f"tv_{k}": v for k, v in tv.items() if k != "demo"},
             **{f"te_{k}": v for k, v in te.items() if k != "demo"})
    shutil.copy(tmp, CACHE_FILE); tmp.unlink()
    print(f"cache written to {CACHE_FILE} ({CACHE_FILE.stat().st_size / 2**20:.0f} MB) in {(time.time() - t0) / 60:.1f} min")
    if CLEAN_RAW_AFTER_CACHE:                                  # free ~13 GB of local disk
        for p in DATA_DIR.glob("*.zip"):
            p.unlink()
        for d in ["ISIC2018_Task1-2_Training_Input", "ISIC2018_Task1-2_Test_Input", "ISIC2018_Task1-2_Validation_Input"]:
            shutil.rmtree(DATA_DIR / d, ignore_errors=True)

_c = np.load(CACHE_FILE)
X_tv, Y_tv, F_tv, SZ_tv, HAIR_tv, ID_tv = (_c[f"tv_{k}"] for k in ["imgs", "masks", "fovs", "sizes", "hair", "ids"])
X_te, Y_te, F_te, SZ_te, HAIR_te, ID_te = (_c[f"te_{k}"] for k in ["imgs", "masks", "fovs", "sizes", "hair", "ids"])
DEMO_RAW = _c["demo_raw"]

tr_idx, va_idx = train_test_split(np.arange(len(X_tv)), test_size=VAL_FRACTION, random_state=SEED, shuffle=True)
print(f"train {len(tr_idx)} | val {len(va_idx)} | test {len(X_te)} | images {X_tv.shape[1:]} | "
      f"hair removed in {HAIR_tv.mean() * 100:.1f}% of train images, {HAIR_te.mean() * 100:.1f}% of test images")
print("NB: val is a random split of the training pool (HAM10000 has several photos per lesion -> val scores are optimistic); "
      "the official test set is the number to trust.")

In [ ]:
# ---- what did the medical preprocessing do?  (re-computed on a few cached raw samples that contain hair)
if len(DEMO_RAW):
    n = min(4, len(DEMO_RAW))
    fig, ax = plt.subplots(n, 4, figsize=(13, 3.3 * n), squeeze=False)
    for r in range(n):
        raw = DEMO_RAW[r]; fov = compute_fov_mask(raw)
        clean, hair_mask, _ = remove_hair(raw, fov)
        proc, _, info = preprocess_dermoscopy(raw)
        for c, (im, title) in enumerate([(raw, "raw (resized)"), (hair_mask * 255, f"detected hair ({info['hair_frac'] * 100:.1f}%)"),
                                         (clean, "hair removed"), (proc, "+ colour constancy + CLAHE")]):
            ax[r, c].imshow(im, cmap="gray" if im.ndim == 2 else None); ax[r, c].set_title(title, fontsize=9); ax[r, c].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("no hairy demo samples stored in the cache")

In [ ]:
# ============================ 8. DATASETS, AUGMENTATION, DATALOADERS ============================
IM_MEAN, IM_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def lab_color_transfer(img, ref, rng=random):
    """Partial Reinhard colour transfer in LAB (skin-tone / device robustness). Blend factor ~ U(0.3, 0.8)."""
    a = rng.uniform(0.3, 0.8)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(np.float32)
    rlab = cv2.cvtColor(ref, cv2.COLOR_RGB2LAB).astype(np.float32)
    m, s = lab.mean((0, 1)), lab.std((0, 1)) + 1e-6
    m2, s2 = rlab.mean((0, 1)), rlab.std((0, 1)) + 1e-6
    mt, st = m + a * (m2 - m), s + a * (s2 - s)
    out = (lab - m) / s * st + mt
    return cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)

norm_tf = A.Compose([A.Normalize(mean=IM_MEAN, std=IM_STD), ToTensorV2()])
train_tf = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),          # dermoscopy has no canonical orientation
    A.Affine(scale=(0.85, 1.15), translate_percent=(-0.06, 0.06), rotate=(-30, 30), p=0.6),
    A.ElasticTransform(alpha=80, sigma=8, p=0.15),
    A.HueSaturationValue(hue_shift_limit=6, sat_shift_limit=15, val_shift_limit=10, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.OneOf([A.MedianBlur(blur_limit=(3, 5)), A.MotionBlur(blur_limit=(3, 7))], p=0.15),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(0.04, 0.12), hole_width_range=(0.04, 0.12), p=0.2),   # default fill = 0 (black)
    A.Normalize(mean=IM_MEAN, std=IM_STD), ToTensorV2(),
])

class ISICCacheDataset(Dataset):
    def __init__(self, imgs, masks, idx, train=False, ct_prob=0.3):
        self.imgs, self.masks, self.idx, self.train, self.ct_prob = imgs, masks, np.asarray(idx), train, ct_prob
        self.aug = train_tf if train else norm_tf
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; img, mask = self.imgs[j], self.masks[j]
        if self.train and random.random() < self.ct_prob:
            img = lab_color_transfer(img, self.imgs[random.choice(self.idx)])
        out = self.aug(image=img, mask=mask)
        return out["image"], out["mask"].float().unsqueeze(0)

def seed_worker(worker_id):
    """Albumentations 2.x keeps its own RNG: re-seed it per worker, otherwise every worker repeats the same augmentations."""
    ws = torch.initial_seed() % 2**32
    np.random.seed(ws); random.seed(ws)
    ds = torch.utils.data.get_worker_info().dataset
    if hasattr(ds.aug, "set_random_seed"):
        ds.aug.set_random_seed(ws)

train_ds = ISICCacheDataset(X_tv, Y_tv, tr_idx, train=True)
val_ds   = ISICCacheDataset(X_tv, Y_tv, va_idx, train=False)
_dl_kw = dict(num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker, persistent_workers=NUM_WORKERS > 0,
              multiprocessing_context="fork" if NUM_WORKERS > 0 else None)      # explicit: notebook-defined classes cannot be pickled for spawn/forkserver
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **_dl_kw)
val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, **_dl_kw)
print(f"{len(train_dl)} train steps / epoch (batch {BATCH_SIZE}) | {len(val_dl)} val steps")

# quick visual sanity check of the augmentation pipeline (same lesion, 5 random draws)
_den = lambda t: (t.permute(1, 2, 0).numpy() * np.array(IM_STD) + np.array(IM_MEAN)).clip(0, 1)
fig, ax = plt.subplots(2, 5, figsize=(14, 6))
for c in range(5):
    im, mk = train_ds[0]
    ax[0, c].imshow(_den(im)); ax[1, c].imshow(mk[0], cmap="gray"); ax[0, c].axis("off"); ax[1, c].axis("off")
plt.suptitle("augmented samples (top) and their masks (bottom)"); plt.tight_layout(); plt.show()

In [ ]:
# ============================ 9. SKINMAMBA (built from the paper, on top of the mamba-ssm kernel) ============================
# Architecture (Zou et al., 2024):  Init-Conv(3->C) -> 5 x [Encoder block -> Down] -> FBGM -> 5 x [Up -> Decoder block] -> 1x1 head
#   Encoder block = SRSSB -> 3x3 conv-BN-ReLU (output is the skip)       Decoder block = fuse(skip) -> 3x3 conv-BN-ReLU -> SRSSB
#   SRSSB  : X' = X + VSSB(LN(X));  X = X' + SMFFL(LN(X'))        (VSSB = linear -> [DWConv-SiLU-SS2D-LN] * SiLU gate -> linear)
#   FBGM   : K' = K + FFGML(LN(K)); K = K' + CSFFL(LN(K'))        (FFGML = FFT -> PW-ReLU-PW -> IFFT -> sigmoid gate)
class LN2d(nn.Module):
    """LayerNorm over the channel axis of a (B,C,H,W) tensor."""
    def __init__(self, dim):
        super().__init__(); self.ln = nn.LayerNorm(dim)
    def forward(self, x):
        return self.ln(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

class SS2DCore(nn.Module):
    """2-D selective scan: the feature map is scanned along 4 directions (row-major, column-major and both reversed)
    with an input-dependent SSM (Mamba's S6), then the 4 sequences are merged back."""
    K = 4
    def __init__(self, d_inner, d_state=16, dt_rank=1, dt_min=1e-3, dt_max=0.1, dt_floor=1e-4):
        super().__init__()
        self.d_inner, self.d_state, self.dt_rank, K = d_inner, d_state, dt_rank, self.K
        self.x_proj_weight = nn.Parameter(torch.stack(
            [nn.Linear(d_inner, dt_rank + 2 * d_state, bias=False).weight.detach() for _ in range(K)]))     # (K, R+2N, D)
        dts = [self._dt_init(dt_rank, d_inner, dt_min, dt_max, dt_floor) for _ in range(K)]
        self.dt_projs_weight = nn.Parameter(torch.stack([w for w, _ in dts]))                                # (K, D, R)
        self.dt_projs_bias = nn.Parameter(torch.stack([b for _, b in dts]))                                  # (K, D)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).repeat(K * d_inner, 1)            # S4D-real init
        self.A_logs = nn.Parameter(torch.log(A))                                                             # (K*D, N)
        self.Ds = nn.Parameter(torch.ones(K * d_inner))                                                      # (K*D,)
        self.A_logs._no_weight_decay = True; self.Ds._no_weight_decay = True

    @staticmethod
    def _dt_init(dt_rank, d_inner, dt_min, dt_max, dt_floor):
        lin = nn.Linear(dt_rank, d_inner, bias=True)
        nn.init.uniform_(lin.weight, -dt_rank ** -0.5, dt_rank ** -0.5)
        dt = torch.exp(torch.rand(d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=dt_floor)
        with torch.no_grad():
            lin.bias.copy_(dt + torch.log(-torch.expm1(-dt)))          # inverse softplus so that softplus(bias) = dt
        return lin.weight.detach(), lin.bias.detach()

    def forward(self, x):                                              # x: (B, D, H, W)  ->  (B, H, W, D)
        B, D, H, W = x.shape; L = H * W; K = self.K
        xs = torch.cat([x.reshape(B, 1, D, L), x.transpose(2, 3).reshape(B, 1, D, L)], dim=1)     # (B,2,D,L)
        xs = torch.cat([xs, torch.flip(xs, dims=[-1])], dim=1)                                      # (B,4,D,L)
        x_dbl = torch.einsum("bkdl,kcd->bkcl", xs, self.x_proj_weight)
        dts, Bs, Cs = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=2)
        dts = torch.einsum("bkrl,kdr->bkdl", dts, self.dt_projs_weight)
        y = selective_scan_fn(
            xs.float().reshape(B, K * D, L), dts.float().contiguous().reshape(B, K * D, L),
            -torch.exp(self.A_logs.float()), Bs.float().contiguous(), Cs.float().contiguous(), self.Ds.float(),
            z=None, delta_bias=self.dt_projs_bias.float().reshape(-1), delta_softplus=True, return_last_state=False,
        ).reshape(B, K, D, L)
        y_rev = torch.flip(y[:, 2:4], dims=[-1])                                                    # undo the reversal
        y_wh = y[:, 1].reshape(B, D, W, H).transpose(2, 3).reshape(B, D, L)                         # undo the transpose
        y_rev_wh = y_rev[:, 1].reshape(B, D, W, H).transpose(2, 3).reshape(B, D, L)
        out = y[:, 0] + y_rev[:, 0] + y_wh + y_rev_wh
        return out.transpose(1, 2).reshape(B, H, W, D)

class VSSB(nn.Module):
    """Visual State Space Block (paper Eq. 3-6). Channels-last (B,H,W,C) in and out."""
    def __init__(self, dim, d_state=16, expand=2, d_conv=3):
        super().__init__()
        d_inner = int(expand * dim)
        self.in_proj = nn.Linear(dim, 2 * d_inner, bias=False)                 # F1 = F2 = Linear(x)  (two streams)
        self.dwconv = nn.Conv2d(d_inner, d_inner, d_conv, padding=d_conv // 2, groups=d_inner)
        self.act = nn.SiLU()
        self.ss2d = SS2DCore(d_inner, d_state, dt_rank=math.ceil(dim / 16))
        self.out_norm = nn.LayerNorm(d_inner)
        self.out_proj = nn.Linear(d_inner, dim, bias=False)
    def forward(self, x):
        f1, f2 = self.in_proj(x).chunk(2, dim=-1)
        f1 = self.act(self.dwconv(f1.permute(0, 3, 1, 2).contiguous()))        # stream 1: DWConv -> SiLU
        f1 = self.out_norm(self.ss2d(f1))                                      #           -> SS2D -> LN
        return self.out_proj(f1 * self.act(f2))                                # stream 2: SiLU gate; multiply; linear

class SMFFL(nn.Module):
    """Scale-Mixed Feed-Forward Layer (Eq. 7-11): two low-dim branches (Linear + 3x3 / 5x5 conv), concat, GELU, Linear.
    The paper's LN + residual (Eq. 2 and Eq. 7/10/11 describe the same path) live in SRSSB, so the residual is applied once."""
    def __init__(self, dim, branch_ratio=0.5):
        super().__init__()
        b = max(8, int(dim * branch_ratio))
        self.fc3, self.fc5 = nn.Linear(dim, b), nn.Linear(dim, b)
        self.conv3, self.conv5 = nn.Conv2d(b, b, 3, padding=1), nn.Conv2d(b, b, 5, padding=2)
        self.fc_out = nn.Linear(2 * b, dim)
    def forward(self, x):                                                      # (B,H,W,C)
        a = self.conv3(self.fc3(x).permute(0, 3, 1, 2))
        c = self.conv5(self.fc5(x).permute(0, 3, 1, 2))
        return self.fc_out(F.gelu(torch.cat([a, c], dim=1).permute(0, 2, 3, 1)))

class SRSSLayer(nn.Module):
    def __init__(self, dim, d_state, expand):
        super().__init__()
        self.norm1, self.vssb = nn.LayerNorm(dim), VSSB(dim, d_state, expand)
        self.norm2, self.smffl = nn.LayerNorm(dim), SMFFL(dim)
    def forward(self, x):
        x = x + self.vssb(self.norm1(x))                                       # Eq. 1
        return x + self.smffl(self.norm2(x))                                   # Eq. 2

class SRSSB(nn.Module):
    """Scale Residual State Space Block, (B,C,H,W) in / out."""
    def __init__(self, dim, depth=1, d_state=16, expand=2):
        super().__init__()
        self.layers = nn.ModuleList([SRSSLayer(dim, d_state, expand) for _ in range(depth)])
    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        for layer in self.layers:
            x = layer(x)
        return x.permute(0, 3, 1, 2).contiguous()

class ConvBNReLU(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class EncoderBlock(nn.Module):
    def __init__(self, dim, depth, d_state, expand):
        super().__init__(); self.srssb, self.conv = SRSSB(dim, depth, d_state, expand), ConvBNReLU(dim, dim)
    def forward(self, x):
        return self.conv(self.srssb(x))

class DecoderBlock(nn.Module):
    def __init__(self, dim, depth, d_state, expand, skip_mode):
        super().__init__()
        self.skip_mode = skip_mode
        self.conv = ConvBNReLU(dim * (2 if skip_mode == "concat" else 1), dim)
        self.srssb = SRSSB(dim, depth, d_state, expand)
    def forward(self, x, skip):
        x = x + skip if self.skip_mode == "add" else torch.cat([x, skip], dim=1)
        return self.srssb(self.conv(x))

class FFGML(nn.Module):
    """Frequency Feature Guidance Modulation Layer (Eq. 14-16): FFT -> PW-conv/ReLU/PW-conv on (real, imag) -> IFFT -> sigmoid gate."""
    def __init__(self, dim):
        super().__init__()
        self.pw1, self.pw2 = nn.Conv2d(2 * dim, 2 * dim, 1), nn.Conv2d(2 * dim, 2 * dim, 1)
    def forward(self, x):
        H, W = x.shape[-2:]
        with torch.autocast(device_type="cuda", enabled=False):                # cuFFT: keep fp32 (and any H, W)
            xf = torch.fft.rfft2(x.float(), norm="ortho")
            z = self.pw2(F.relu(self.pw1(torch.cat([xf.real, xf.imag], dim=1))))
            re, im = z.chunk(2, dim=1)
            gate = torch.sigmoid(torch.fft.irfft2(torch.complex(re, im), s=(H, W), norm="ortho"))
        return gate.to(x.dtype) * x

class CSFFL(nn.Module):
    """Compact Simple Feed-Forward Layer (Eq. 17): 3x3 conv (expand) -> GELU -> 1x1 conv (restore)."""
    def __init__(self, dim, ratio=2):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(dim, dim * ratio, 3, padding=1), nn.GELU(), nn.Conv2d(dim * ratio, dim, 1))
    def forward(self, x):
        return self.net(x)

class FBGM(nn.Module):
    """Frequency Boundary Guided Module (Eq. 12-13), placed between encoder and decoder (Sec. 3.1)."""
    def __init__(self, dim, depth=1):
        super().__init__()
        self.layers = nn.ModuleList([nn.ModuleList([LN2d(dim), FFGML(dim), LN2d(dim), CSFFL(dim)]) for _ in range(depth)])
    def forward(self, x):
        for n1, ffgml, n2, csffl in self.layers:
            x = x + ffgml(n1(x))
            x = x + csffl(n2(x))
        return x

class SkinMamba(nn.Module):
    def __init__(self, in_ch=3, num_classes=1, base=16, depths=(1, 1, 1, 1, 1), d_state=16, expand=2, skip_mode="add", fbgm_depth=1):
        super().__init__()
        dims = [base * 2 ** i for i in range(5)]                                 # 16, 32, 64, 128, 256 (bottleneck: 512)
        self.init_conv = ConvBNReLU(in_ch, base)
        self.enc = nn.ModuleList([EncoderBlock(dims[i], depths[i], d_state, expand) for i in range(5)])
        self.down = nn.ModuleList([nn.Sequential(nn.Conv2d(dims[i], dims[i] * 2, 3, stride=2, padding=1, bias=False),
                                                 nn.BatchNorm2d(dims[i] * 2)) for i in range(5)])
        self.fbgm = FBGM(dims[-1] * 2, fbgm_depth)
        self.up = nn.ModuleList([nn.ConvTranspose2d(dims[i] * 2, dims[i], 2, stride=2) for i in reversed(range(5))])
        self.dec = nn.ModuleList([DecoderBlock(dims[i], depths[i], d_state, expand, skip_mode) for i in reversed(range(5))])
        self.head = nn.Conv2d(base, num_classes, 1)
    def forward(self, x):
        assert x.shape[-1] % 32 == 0 and x.shape[-2] % 32 == 0, "input size must be divisible by 32"
        x, skips = self.init_conv(x), []
        for enc, down in zip(self.enc, self.down):
            x = enc(x); skips.append(x); x = down(x)
        x = self.fbgm(x)
        for up, dec, skip in zip(self.up, self.dec, reversed(skips)):
            x = dec(up(x), skip)
        return self.head(x)                                                      # logits (B, 1, H, W)

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6
print(f"SkinMamba (C={BASE_CHANNELS}) defined: {count_params(SkinMamba(base=BASE_CHANNELS, depths=DEPTHS, d_state=D_STATE, expand=EXPAND, skip_mode=SKIP_MODE, fbgm_depth=FBGM_DEPTH)):.2f} M parameters")

In [ ]:
# ============================ 10. SMOKE TEST: forward/backward, memory, speed ============================
if MAMBA_BACKEND.startswith("pure") and not ALLOW_SLOW_FALLBACK:
    raise RuntimeError("mamba-ssm could not be installed, so only the pure-PyTorch scan is available (far too slow/large for "
                       f"{IMG_SIZE}x{IMG_SIZE}). Re-run the Mamba cell (set BUILD_MAMBA_FROM_SOURCE=True) or set ALLOW_SLOW_FALLBACK=True "
                       "together with a tiny IMG_SIZE (e.g. 64) just to debug.")
_net = SkinMamba(base=BASE_CHANNELS, depths=DEPTHS, d_state=D_STATE, expand=EXPAND, skip_mode=SKIP_MODE, fbgm_depth=FBGM_DEPTH).cuda()
try:
    _x = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device="cuda")
    torch.cuda.reset_peak_memory_stats()
    for it in range(4):
        if it == 1:
            torch.cuda.synchronize(); _t0 = time.time()
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            _y = _net(_x)
        _y.float().mean().backward()
    torch.cuda.synchronize(); _dt = (time.time() - _t0) / 3
    assert _y.shape == (BATCH_SIZE, 1, IMG_SIZE, IMG_SIZE) and torch.isfinite(_y).all(), "bad output"
    print(f"output {tuple(_y.shape)} OK | peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB | "
          f"{_dt:.2f} s/iter -> ~{_dt * len(train_dl) / 60:.1f} min per epoch (rough estimate)")
except torch.cuda.OutOfMemoryError:
    print("OOM: lower BATCH_SIZE (section 1) or IMG_SIZE and re-run")
finally:
    del _net; torch.cuda.empty_cache()

In [ ]:
# ============================ 11. LOSS ============================
class FocalTverskyLoss(nn.Module):
    """alpha < beta -> false negatives cost more than false positives (recall focus, helps small / faint lesions)."""
    def __init__(self, alpha=0.3, beta=0.7, gamma=1.0, eps=1e-6):
        super().__init__(); self.a, self.b, self.g, self.eps = alpha, beta, gamma, eps
    def forward(self, logits, y):
        p, y = torch.sigmoid(logits), y.float()
        tp, fp, fn = (p * y).sum((2, 3)), (p * (1 - y)).sum((2, 3)), ((1 - p) * y).sum((2, 3))
        t = (tp + self.eps) / (tp + self.a * fp + self.b * fn + self.eps)
        return (1.0 - t).pow(self.g).mean()

class StructureLoss(nn.Module):
    """Edge-weighted BCE + edge-weighted soft IoU (PraNet style). Weights are large on the lesion boundary."""
    def __init__(self, img_size=224, edge_w=5.0):
        super().__init__()
        self.k = max(3, int(round(31 * img_size / 512)) | 1)           # 31 px window at 512 px, scaled to our resolution
        self.edge_w = edge_w
    def forward(self, logits, mask):
        mask = mask.float()
        with torch.no_grad():
            weit = 1.0 + self.edge_w * (F.avg_pool2d(mask, self.k, stride=1, padding=self.k // 2) - mask).abs()
        wbce = (weit * F.binary_cross_entropy_with_logits(logits, mask, reduction="none")).sum((2, 3)) / weit.sum((2, 3))
        p = torch.sigmoid(logits)
        inter, union = ((p * mask) * weit).sum((2, 3)), ((p + mask) * weit).sum((2, 3))
        return (wbce + (1.0 - (inter + 1.0) / (union - inter + 1.0))).mean()

class ComboLoss(nn.Module):
    def __init__(self, img_size=224, ft_weight=0.6):
        super().__init__(); self.ft, self.st, self.w = FocalTverskyLoss(), StructureLoss(img_size), ft_weight
    def forward(self, logits, y):
        logits = logits.float()                                         # compute the loss in fp32 even under AMP
        return self.w * self.ft(logits, y) + (1.0 - self.w) * self.st(logits, y)

In [ ]:
# ============================ 12. LIGHTNING MODULE ============================
class LitSkinMamba(pl.LightningModule):
    def __init__(self, base=16, depths=(1, 1, 1, 1, 1), d_state=16, expand=2, skip_mode="add", fbgm_depth=1,
                 lr=1e-3, weight_decay=1e-4, max_epochs=100, warmup_epochs=3, ft_weight=0.6, img_size=224):
        super().__init__()
        self.save_hyperparameters()
        self.net = SkinMamba(base=base, depths=depths, d_state=d_state, expand=expand, skip_mode=skip_mode, fbgm_depth=fbgm_depth)
        self.criterion = ComboLoss(img_size, ft_weight)
        self._reset_val()

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.criterion(self(x), y)
        self.log("train/loss", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=x.size(0))
        return loss

    # -------- validation: paper-style pooled metrics + per-image Dice --------
    def _reset_val(self):
        self._cm = torch.zeros(4, dtype=torch.float64); self._dice_sum = 0.0; self._n = 0

    def on_validation_epoch_start(self):
        self._reset_val()

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        self.log("val/loss", self.criterion(logits, y), on_step=False, on_epoch=True, batch_size=x.size(0))
        p, g = torch.sigmoid(logits.float()) >= 0.5, y >= 0.5
        tp, fp, fn = (p & g).sum((1, 2, 3)).double(), (p & ~g).sum((1, 2, 3)).double(), (~p & g).sum((1, 2, 3)).double()
        tn = float(p[0].numel()) - tp - fp - fn
        self._cm += torch.stack([tp.sum(), fp.sum(), fn.sum(), tn.sum()]).cpu()
        den = 2 * tp + fp + fn
        self._dice_sum += torch.where(den > 0, 2 * tp / den.clamp(min=1), torch.ones_like(den)).sum().item()
        self._n += x.size(0)

    def on_validation_epoch_end(self):
        tp, fp, fn, tn = self._cm.tolist(); e = 1e-9
        self.log_dict({"val/dsc": 2 * tp / (2 * tp + fp + fn + e), "val/miou": tp / (tp + fp + fn + e),
                       "val/sen": tp / (tp + fn + e), "val/spe": tn / (tn + fp + e), "val/acc": (tp + tn) / (tp + fp + fn + tn + e),
                       "val/dice_img": self._dice_sum / max(1, self._n)}, prog_bar=False)
        self.log("val/dsc_bar", 2 * tp / (2 * tp + fp + fn + e), prog_bar=True, logger=False)

    def configure_optimizers(self):
        decay, no_decay = [], []
        for n, p in self.net.named_parameters():
            skip = p.ndim <= 1 or n.endswith(".bias") or n.endswith("dt_projs_bias") or getattr(p, "_no_weight_decay", False)
            (no_decay if skip else decay).append(p)
        opt = torch.optim.AdamW([{"params": decay, "weight_decay": self.hparams.weight_decay},
                                 {"params": no_decay, "weight_decay": 0.0}], lr=self.hparams.lr, betas=(0.9, 0.999))
        warm = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, end_factor=1.0, total_iters=self.hparams.warmup_epochs)
        cos = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, self.hparams.max_epochs - self.hparams.warmup_epochs), eta_min=1e-6)
        sched = torch.optim.lr_scheduler.SequentialLR(opt, [warm, cos], milestones=[self.hparams.warmup_epochs])
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "epoch"}}

In [ ]:
# ============================ 13. TRAIN (auto-resumes from Drive) ============================
pl.seed_everything(SEED, workers=True)
model = LitSkinMamba(base=BASE_CHANNELS, depths=DEPTHS, d_state=D_STATE, expand=EXPAND, skip_mode=SKIP_MODE, fbgm_depth=FBGM_DEPTH,
                     lr=LR, weight_decay=WEIGHT_DECAY, max_epochs=MAX_EPOCHS, warmup_epochs=WARMUP_EPOCHS,
                     ft_weight=FT_WEIGHT, img_size=IMG_SIZE)

ckpt_cb = ModelCheckpoint(dirpath=str(CKPT_DIR), filename="best", monitor="val/dsc", mode="max", save_top_k=1,
                          save_last=True, enable_version_counter=False)       # best.ckpt + last.ckpt (overwritten in place)
callbacks = [ckpt_cb, EarlyStopping(monitor="val/dsc", mode="max", patience=EARLY_STOP_PATIENCE), LearningRateMonitor(logging_interval="epoch")]

loggers = [CSVLogger(save_dir=str(DRIVE_DIR / "logs"), name="skinmamba")]
if USE_WANDB:
    !pip install -q wandb
    import wandb
    from google.colab import userdata                  # add WANDB_API_KEY under the key icon (Secrets) in the left sidebar
    wandb.login(key=userdata.get("WANDB_API_KEY"))
    loggers.append(pl.loggers.WandbLogger(project="Skin-Lesion-Segmentation", name="SkinMamba-ISIC2018", save_dir="/content"))

trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1, precision=PRECISION, gradient_clip_val=1.0,
                     accumulate_grad_batches=ACCUM, callbacks=callbacks, logger=loggers, log_every_n_steps=20,
                     num_sanity_val_steps=0, enable_model_summary=True)

last = CKPT_DIR / "last.ckpt"
RESUME_FROM = str(last) if last.exists() else None
print("resuming from", RESUME_FROM) if RESUME_FROM else print("starting a fresh run")
trainer.fit(model, train_dl, val_dl, ckpt_path=RESUME_FROM)
print("best checkpoint:", ckpt_cb.best_model_path, "| best val DSC:", ckpt_cb.best_model_score)

In [ ]:
# ============================ 14. TRAINING CURVES (all sessions concatenated) ============================
_logs = sorted((DRIVE_DIR / "logs" / "skinmamba").glob("version_*/metrics.csv"), key=lambda p: int(p.parent.name.split("_")[1]))
if _logs:
    hist = pd.concat([pd.read_csv(p) for p in _logs], ignore_index=True).groupby("epoch").mean(numeric_only=True)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    for c in ["train/loss", "val/loss"]:
        if c in hist: ax[0].plot(hist.index, hist[c], label=c)
    ax[0].set_title("loss"); ax[0].legend()
    for c in ["val/dsc", "val/miou", "val/dice_img"]:
        if c in hist: ax[1].plot(hist.index, hist[c], label=c)
    ax[1].set_title("validation overlap metrics"); ax[1].legend()
    for c in [c for c in hist.columns if c.startswith("lr-")]:
        ax[2].plot(hist.index, hist[c], label=c)
    ax[2].set_title("learning rate"); ax[2].legend()
    for a in ax: a.set_xlabel("epoch"); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print(hist[["val/dsc", "val/miou"]].tail(3))
else:
    print("no logs yet - run the training cell first")

In [ ]:
# ============================ 15. POST-PROCESSING + METRICS ============================
# ---------------------------------------------------------------------------
# Post-processing (medical priors) + metrics.  Pure numpy / OpenCV / SciPy.
#   * FOV gating            : lesions cannot lie in the black dermoscope vignette
#   * morphological open+close, hole filling : smooth spurs, fill specular holes / hair gaps
#   * drop tiny components  : keep every component >= 10 % of the largest one (satellite lesions survive)
#   * optional DenseCRF     : colour-aware boundary refinement (only if pydensecrf is installed)
# ---------------------------------------------------------------------------
def postprocess_mask(prob, fov=None, thr=0.5, min_rel_area=0.10, ref_size=224):
    raw = (prob >= thr).astype(np.uint8)
    m = raw & fov if fov is not None else raw.copy()
    r = max(1, int(round(2 * max(m.shape) / ref_size)))
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
    m = binary_fill_holes(m).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n > 1:
        areas = st[1:, cv2.CC_STAT_AREA]
        m = np.isin(lab, np.flatnonzero(areas >= min_rel_area * areas.max()) + 1).astype(np.uint8)
    return m if m.any() else raw                         # never delete a (tiny) lesion completely

try:
    import pydensecrf.densecrf as dcrf
    from pydensecrf.utils import unary_from_softmax
    HAS_CRF = True
except Exception:
    HAS_CRF = False

def crf_refine(img, prob, iters=5, sxy_g=3, compat_g=3, sxy_b=40, srgb=13, compat_b=8):
    """DenseCRF on the lesion probability (img: HxWx3 uint8, prob: HxW float in [0,1]). Returns uint8 mask."""
    H, W = prob.shape
    p = np.clip(prob, 1e-4, 1 - 1e-4)
    d = dcrf.DenseCRF2D(W, H, 2)
    d.setUnaryEnergy(np.ascontiguousarray(unary_from_softmax(np.stack([1 - p, p]).astype(np.float32))))
    d.addPairwiseGaussian(sxy=sxy_g, compat=compat_g)
    d.addPairwiseBilateral(sxy=sxy_b, srgb=srgb, rgbim=np.ascontiguousarray(img, dtype=np.uint8), compat=compat_b)
    Q = np.array(d.inference(iters)).reshape(2, H, W)
    return Q.argmax(0).astype(np.uint8)

def upsample_mask(mask, size_hw):
    """Bilinear up-sampling of a binary mask, re-thresholded at 0.5 -> smooth contours at full resolution."""
    h0, w0 = size_hw
    return (cv2.resize(mask.astype(np.float32), (w0, h0), interpolation=cv2.INTER_LINEAR) > 0.5).astype(np.uint8)

# ------------------------------- metrics ---------------------------------
def seg_counts(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    tp = int((p & g).sum()); fp = int((p & ~g).sum()); fn = int((~p & g).sum())
    return tp, fp, fn, p.size - tp - fp - fn

def aggregate_metrics(counts):
    """counts: (N,4) array of [TP,FP,FN,TN].  Returns paper-style (micro, pooled over all pixels) metrics and
    per-image (macro) metrics, incl. the official ISIC-2018 'thresholded Jaccard' (score 0 if Jaccard < 0.65)."""
    c = np.asarray(counts, dtype=np.float64)
    tp, fp, fn, tn = c.sum(0)
    out = {"mIoU": tp / (tp + fp + fn), "DSC": 2 * tp / (2 * tp + fp + fn), "Acc": (tp + tn) / c.sum(),
           "Spe": tn / (tn + fp), "Sen": tp / (tp + fn)}
    d = c[:, 0] + c[:, 1] + c[:, 2]
    jac = np.where(d > 0, c[:, 0] / np.maximum(d, 1), 1.0)
    dice = np.where(d > 0, 2 * c[:, 0] / np.maximum(2 * c[:, 0] + c[:, 1] + c[:, 2], 1), 1.0)
    out.update({"Jaccard/img": jac.mean(), "Dice/img": dice.mean(), "Jaccard_thr0.65": np.where(jac >= 0.65, jac, 0.0).mean()})
    return {k: float(v) for k, v in out.items()}

In [ ]:
# ============================ 16. INFERENCE: TTA + POST-PROCESSING + ABLATION ============================
# (Fresh runtime? Run sections 1-12, skip the training section, then this one: it loads best.ckpt from Drive.)
best_path = CKPT_DIR / "best.ckpt"
assert best_path.exists(), f"{best_path} not found - train first (or point best_path to a checkpoint)"
lit = LitSkinMamba.load_from_checkpoint(best_path, map_location="cpu").cuda().eval()
net = lit.net
MEAN_T = torch.tensor(IM_MEAN, device="cuda").view(1, 3, 1, 1); STD_T = torch.tensor(IM_STD, device="cuda").view(1, 3, 1, 1)

@torch.inference_mode()
def predict_probs(imgs, tta=False, bs=None):
    """imgs: (N,H,W,3) uint8 (already preprocessed) -> (N,H,W) float32 lesion probabilities. TTA = 4 flips."""
    bs = bs or BATCH_SIZE * 2
    out = np.zeros(imgs.shape[:3], np.float32)
    for i in range(0, len(imgs), bs):
        x = torch.from_numpy(imgs[i:i + bs]).cuda(non_blocking=True).permute(0, 3, 1, 2).float().div_(255)
        x = (x - MEAN_T) / STD_T
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            p = torch.sigmoid(net(x).float())
            if tta:
                p = p + torch.sigmoid(net(x.flip(-1)).float()).flip(-1)
                p = p + torch.sigmoid(net(x.flip(-2)).float()).flip(-2)
                p = p + torch.sigmoid(net(x.flip(-1, -2)).float()).flip(-1, -2)
                p = p / 4
        out[i:i + bs] = p[:, 0].cpu().numpy()
    return out

X_va, Y_va, F_va = X_tv[va_idx], Y_tv[va_idx], F_tv[va_idx]
SPLITS = {"val": (X_va, Y_va, F_va), "test": (X_te, Y_te, F_te)}
P = {s: {"plain": predict_probs(X), "tta": predict_probs(X, tta=True)} for s, (X, _, _) in SPLITS.items()}

def eval_masks(masks, gts):
    return aggregate_metrics([seg_counts(m, g) for m, g in zip(masks, gts)])

def post_all(probs, fovs, thr=0.5):
    return [postprocess_mask(p, f, thr) for p, f in zip(probs, fovs)]

# threshold tuned on the validation split only (never on test)
_sweep = {round(t, 2): eval_masks(post_all(P["val"]["tta"], F_va, t), Y_va)["DSC"] for t in np.arange(0.30, 0.71, 0.05)}
BEST_THR = max(_sweep, key=_sweep.get)
print("val DSC vs threshold:", {k: round(v, 4) for k, v in _sweep.items()}, "-> BEST_THR =", BEST_THR)

def ablation(split):
    X, Y, Fv = SPLITS[split]; p0, p1 = P[split]["plain"], P[split]["tta"]
    rows = {"raw (thr 0.5)": eval_masks(p0 >= 0.5, Y),
            "+ flip-TTA": eval_masks(p1 >= 0.5, Y),
            "+ TTA + post-processing": eval_masks(post_all(p1, Fv), Y)}
    if HAS_CRF:
        rows["+ TTA + DenseCRF + post-proc"] = eval_masks(
            [postprocess_mask(crf_refine(x, p).astype(np.float32), f) for x, p, f in zip(X, p1, Fv)], Y)
    rows[f"+ TTA + post-proc, thr {BEST_THR} (val-tuned)"] = eval_masks(post_all(p1, Fv, BEST_THR), Y)
    return (pd.DataFrame(rows).T * 100).round(2)

for split in ["val", "test"]:
    print(f"\n===== {split.upper()} @ {IMG_SIZE}x{IMG_SIZE}  (all values in %) =====")
    display(ablation(split))
print("\nHAS_CRF =", HAS_CRF, "(optional: !pip install git+https://github.com/lucasb-eyer/pydensecrf.git, then re-run sections 15-16)")

In [ ]:
# ============================ 17. FULL-RESOLUTION EXPORT + OFFICIAL-STYLE EVALUATION ============================
# Predictions are post-processed at network resolution, up-sampled to each image's ORIGINAL size, saved as ISIC-style
# PNGs (ISIC_xxxxxxx_segmentation.png, 0/255) and scored against the original-resolution test ground truth.
def export_fullres(thr, out_dir=PRED_DIR):
    out_dir.mkdir(parents=True, exist_ok=True); counts = []
    for i in tqdm(range(len(ID_te)), desc="export"):
        m = postprocess_mask(P["test"]["tta"][i], F_te[i], thr)
        h0, w0 = (int(v) for v in SZ_te[i])
        full = upsample_mask(m, (h0, w0))
        cv2.imwrite(str(out_dir / f"{ID_te[i]}_segmentation.png"), full * 255)
        gt = cv2.imread(str(TEST_GT_DIR / f"{ID_te[i]}_segmentation.png"), cv2.IMREAD_GRAYSCALE)
        if gt.shape != full.shape:
            gt = cv2.resize(gt, (w0, h0), interpolation=cv2.INTER_NEAREST)
        counts.append(seg_counts(full, gt > 127))
    return aggregate_metrics(counts)

final = export_fullres(BEST_THR)
print("\nOFFICIAL TEST SET (1000 images), original resolution, val-tuned threshold", BEST_THR)
print(pd.Series(final).mul(100).round(2).to_string())
print("\n'Jaccard_thr0.65' is the official ISIC-2018 Task-1 ranking metric; mIoU/DSC/Acc/Spe/Sen are pooled over all pixels like the paper's tables.")
print("NB: the paper evaluates a random 70/30 split of 2694 images; here the test set is the official held-out one, so numbers are not directly comparable.")

zip_path = shutil.make_archive(str(DRIVE_DIR / "SkinMamba_ISIC2018_Task1_predictions"), "zip", PRED_DIR)
torch.save(net.state_dict(), DRIVE_DIR / "skinmamba_best_weights.pt")
print("saved:", zip_path, "and", DRIVE_DIR / "skinmamba_best_weights.pt")

In [ ]:
# ============================ 18. QUALITATIVE RESULTS + SEGMENT ANY DERMOSCOPY IMAGE ============================
_masks = post_all(P["test"]["tta"], F_te, BEST_THR)
_cnt = np.array([seg_counts(m, g) for m, g in zip(_masks, Y_te)], dtype=np.float64)
_jac = _cnt[:, 0] / np.maximum(_cnt[:, 0] + _cnt[:, 1] + _cnt[:, 2], 1)

def overlay(img, gt, pred):
    out = img.copy()
    for m, col in [(gt, (0, 255, 0)), (pred, (255, 0, 0))]:                     # green = ground truth, red = prediction
        cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(out, cs, -1, col, 1)
    return out

rng_ = np.random.default_rng(SEED)
picks = list(rng_.choice(len(X_te), 6, replace=False)) + list(np.argsort(_jac)[:3])
fig, ax = plt.subplots(3, 3, figsize=(13, 13))
for a, i in zip(ax.ravel(), picks):
    a.imshow(overlay(X_te[i], Y_te[i], _masks[i])); a.axis("off")
    a.set_title(f"{ID_te[i]}  Jaccard {_jac[i]:.3f}" + ("  (worst-case)" if i in picks[6:] else ""), fontsize=9)
plt.suptitle("preprocessed test images | green = ground truth, red = SkinMamba | last row = 3 worst test cases"); plt.tight_layout(); plt.show()

def segment_image(path_or_rgb, thr=None, tta=True):
    """End-to-end inference for ONE new dermoscopy image: resize -> medical preprocessing -> model (+TTA) -> post-processing
    -> mask at the image's original resolution (uint8 0/1)."""
    thr = BEST_THR if thr is None else thr
    if isinstance(path_or_rgb, (str, Path)):
        raw, (h0, w0) = load_resized_rgb(path_or_rgb, S)
    else:
        h0, w0 = path_or_rgb.shape[:2]; raw = cv2.resize(path_or_rgb, (S, S), interpolation=cv2.INTER_AREA)
    proc, fov, _ = preprocess_dermoscopy(raw)
    prob = predict_probs(proc[None], tta=tta)[0]
    return upsample_mask(postprocess_mask(prob, fov, thr), (h0, w0)), prob

# demo on one of the original test images (re-download the test input zip first if you cleaned the raw data)
_demo = next(iter((DATA_DIR / "ISIC2018_Task1-2_Test_Input").glob("*.jpg")), None) if (DATA_DIR / "ISIC2018_Task1-2_Test_Input").exists() else None
if _demo is not None:
    mask_full, _ = segment_image(_demo)
    img_full = np.asarray(Image.open(_demo).convert("RGB"))
    plt.figure(figsize=(8, 6)); plt.imshow(overlay(img_full, np.zeros_like(mask_full), mask_full)); plt.axis("off"); plt.title(_demo.name); plt.show()
else:
    print("segment_image(path) is ready to use on any dermoscopy JPEG, e.g.  mask, prob = segment_image('/content/my_image.jpg')")

## Notes, assumptions and knobs

**Where I had to interpret the paper** (the authors' code was deliberately not copied)
* *Skip connections*: Sec. 3.1 says "simple addition", Sec. 3.2 says features are "aggregated along the channel dimension". Default is `SKIP_MODE="add"`; set `"concat"` to try the other reading.
* *FBGM* sits between encoder and decoder at the 1/32-resolution bottleneck (Sec. 3.1). Its depth (1), the CSFFL expansion (×2), SMFFL branch width and the down/up-sampling layers are not specified in the text and were chosen conventionally.
* *SMFFL* residual: Eq. 2 and Eq. 11 describe the same residual path, so it is applied once.
* *Loss, augmentation, preprocessing* go beyond the paper (which only mentions flips/rotations and does not state a loss): Focal-Tversky + Structure loss, the richer augmentation set and the dermoscopy preprocessing/post-processing are this pipeline's additions to "improve the final result". The ablation table (section 16) shows what TTA / post-processing / CRF actually buy on *your* run; ablate preprocessing by training once with the raw resized images if you want that number too.

**Colab specifics**
* Everything that must survive a disconnect lives on Drive (`MyDrive/SkinMamba_ISIC2018`): preprocessed cache, `last.ckpt`/`best.ckpt`, CSV logs, predictions. Re-running the notebook after a disconnect resumes training automatically — don't change `MAX_EPOCHS` mid-run (the cosine schedule is restored from the checkpoint).
* If `mamba-ssm` has no prebuilt wheel for the runtime, section 3 compiles it (10-25 min per fresh session). A different Colab torch/CUDA combination is the usual cause.
* NaN loss on a T4 → set `PRECISION="32-true"` in section 1. OOM → lower `BATCH_SIZE` or `IMG_SIZE` (multiple of 32).
* Weights & Biases: `USE_WANDB=True` reads `WANDB_API_KEY` from Colab *Secrets* (never hard-code keys in a notebook).

**Dataset license:** ISIC 2018 training data is CC-BY-NC 4.0 — cite HAM10000 (Tschandl et al., 2018) and the ISIC 2018 challenge paper (Codella et al., 2019) when you publish results.